In [ ]:
# Configure visualization, evaluation, or cached rejudging.
RUN_MODE = "rejudge"
MODE = "sample"
DATASET = "cnc"
SAMPLE_ID = 5
TRIPLE_SOURCE = "gold"

MANUAL_TEXT = "5 Afghan troops killed after US army bombards warehouse in Kabul"
MANUAL_CAUSE = "US army bombards warehouse in Kabul"
MANUAL_EFFECT = "5 Afghan troops killed"

CAUSAL_PROMPT_VERSION = "v9.2"
EVENT_PROMPT_VERSION = "nested_v1"
USE_RAG = False
RAG_MODE = "knn_pattern"
RAG_TOP_K = 3

EVAL_DATASET = "cnc"
EVAL_SAMPLE_N = 300
EVAL_MAX_SPANS = None
EVAL_SCHEMA = "auto"
EVAL_MAX_DEPTH = "auto"
EVAL_EXTRACTION_RETRY_TIMES = 1
EVAL_CHECKPOINT_EVERY = 100
RUN_JUDGE = True
JUDGE_PROMPT_VERSION = "v3"
JUDGE_API_KEY_PATH = "deepseek_api.txt"
JUDGE_MODEL = "deepseek-v4-pro"
JUDGE_MAX_TOKENS = 2048
JUDGE_TIMEOUT = 60
JUDGE_MAX_WORKERS = 10
JUDGE_RETRY_TIMES = 3
JUDGE_RETRY_BASE_SECONDS = 2

RUN_UNIT_REJUDGE = True

REJUDGE_SOURCE_SPANS_PATH = "results/kg_evaluation/cnc_nested_v1_n300_20260901_132127_spans.jsonl"
REJUDGE_OUTPUT_DIR = "results/kg_evaluation/rejudged"

RUN_GLOBAL_DIAGNOSTICS = True
GLOBAL_DIAGNOSTIC_PROMPT_VERSION = "v1"
GLOBAL_DIAGNOSTIC_MAX_WORKERS = 10
GLOBAL_DIAGNOSTIC_OUTPUT_DIR = "results/kg_evaluation/global_diagnostics"

LLM_BASE_URL = "http://127.0.0.1:1234/v1"
LLM_API_KEY = "lm-studio"
MODEL_NAME = "auto"
TEMPERATURE = 0.0
MAX_TOKENS = 2048
CONTEXT_LENGTH = 8192
LLM_EXTRA_BODY = {}
LLM_TIMEOUT = 120
LLM_RETRY_TIMES = 3


In [ ]:
# Initialize dependencies, clients, and sample helpers.
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError("The active notebook kernel is not Master_thesis; switch kernels and rerun.")

from src.data_io import load_dataset
from src.event_extractor import build_kg_json, extract_event, parse_event_output
from src.generator import generate
from src.kg_serializer import to_rdf
from src.kg_eval_pipeline import (
    KGEvalConfig, KGRejudgeConfig, load_saved_span_results,
    resolve_eval_max_depth, resolve_eval_schema, run_kg_evaluation, run_saved_kg_judging,
)
from src.kg_global_diagnostics import (
    aggregate_global_diagnostics, prepare_global_diagnostic_records,
    run_parallel_global_diagnostics, save_global_diagnostic_outputs,
)
from src.kg_evaluator import (
    DeepSeekJudgeClient,
    aggregate_sample_results,
    build_judge_prompt,
    compute_span_metrics,
    flatten_judge_units,
    load_gold_span_records,
    parse_judge_output,
    run_parallel_judge,
    run_eval_extraction,
    save_eval_checkpoint,
    save_eval_outputs,
    validate_extraction,
)
from src.llm_client import LLMClient
from src.retriever import create_retriever

if RUN_MODE == "rejudge":
    client = None
    retriever = None
    print("Rejudge mode: skip LM Studio connection and reuse saved parsed extractions.")
else:
    client = LLMClient(
        base_url=LLM_BASE_URL,
        model=MODEL_NAME,
        api_key=LLM_API_KEY,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        context_length=CONTEXT_LENGTH,
        extra_body=LLM_EXTRA_BODY,
        timeout=LLM_TIMEOUT,
        retry_times=LLM_RETRY_TIMES,
    )
    if MODEL_NAME == "auto":
        loaded_models = client.list_loaded_models()
        if len(loaded_models) == 1:
            client.model = loaded_models[0]
            model_source = "LM Studio auto"
        elif len(loaded_models) == 0:
            raise RuntimeError("MODEL_NAME='auto', but LM Studio has no loaded chat model.")
        else:
            raise RuntimeError(f"MODEL_NAME='auto', but LM Studio returned multiple models: {loaded_models}. Set MODEL_NAME explicitly.")
    else:
        model_source = "notebook MODEL_NAME"
    retriever = create_retriever(RAG_MODE) if USE_RAG else None
    print(f"Base URL: {client.base_url}")
    print(f"Model: {client.model}")
    print(f"Model source: {model_source}")
print(f"Triple source: {TRIPLE_SOURCE}")
print(f"RUN_MODE: {RUN_MODE}")

def pred_triples_to_kg_triples(prediction):
    if not prediction.get("has_causal", False):
        return []
    return prediction.get("triples", []) if isinstance(prediction.get("triples"), list) else []

def gold_relations_to_kg_triples(sample):
    if not sample.get("has_causal", False):
        return []
    triples = []
    for relation in sample.get("relations", []):
        triples.append({"cause": relation.get("cause", ""), "effect": relation.get("effect", ""), "relation": "caused"})
    return triples

def load_sample_by_id(dataset, sample_id):
    samples = load_dataset(dataset)
    for sample in samples:
        if sample.get("id") == sample_id:
            return sample
    raise ValueError(f"No sample with id={sample_id} was found in {dataset}")

def build_current_kg_json():
    if MODE == "manual":
        triples = [{"cause": MANUAL_CAUSE, "effect": MANUAL_EFFECT, "relation": "caused"}]
        return build_kg_json(
            sample_id=None,
            text=MANUAL_TEXT,
            triples=triples,
            client=client,
            prompt_version=EVENT_PROMPT_VERSION,
            triple_source="manual",
        )

    sample = load_sample_by_id(DATASET, SAMPLE_ID)
    if TRIPLE_SOURCE == "gold":
        triples = gold_relations_to_kg_triples(sample)
        print("Gold relations:")
        print(json.dumps(sample.get("relations", []), indent=2, ensure_ascii=False))
    elif TRIPLE_SOURCE == "pred":
        prediction_key = (DATASET, SAMPLE_ID, CAUSAL_PROMPT_VERSION, USE_RAG, RAG_MODE, RAG_TOP_K)
        if globals().get("demo1_prediction_key") == prediction_key and "demo1_prediction" in globals():
            prediction = demo1_prediction
            print("Using predicted triples from the preceding Demo1 preview cell.")
        else:
            prediction = generate(
                text=sample["text"],
                sample_id=sample["id"],
                client=client,
                retriever=retriever,
                use_rag=USE_RAG,
                top_k=RAG_TOP_K,
                rag_mode=RAG_MODE,
                prompt_name=CAUSAL_PROMPT_VERSION,
            )
        triples = pred_triples_to_kg_triples(prediction)
        print("Gold relations:")
        print(json.dumps(sample.get("relations", []), indent=2, ensure_ascii=False))
        print("Pred triples:")
        print(json.dumps(prediction.get("triples", []), indent=2, ensure_ascii=False))
    else:
        raise ValueError("TRIPLE_SOURCE must be 'gold' or 'pred'")

    if not triples:
        print(f"The current sample has no {TRIPLE_SOURCE} causal triple to graph.")
    return build_kg_json(
        sample_id=sample["id"],
        text=sample["text"],
        triples=triples,
        client=client,
        prompt_version=EVENT_PROMPT_VERSION,
        triple_source=TRIPLE_SOURCE,
    )


In [ ]:
# Preview causal extraction and evaluation for one sample.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:

    from src.evaluator import build_sample_judgement

    if MODE == "manual":
        selected_sample = {
            "id": None,
            "text": MANUAL_TEXT,
            "has_causal": True,
            "relations": [{"cause": MANUAL_CAUSE, "effect": MANUAL_EFFECT}],
        }
        demo1_prediction = {
            "id": None,
            "has_causal": True,
            "triples": [{"cause": {"span": MANUAL_CAUSE}, "relation": "caused", "effect": {"span": MANUAL_EFFECT}}],
        }
        demo1_prediction_key = None
        print("MODE='manual': using the supplied cause/effect pair for the Demo1 preview.")
    else:
        selected_sample = load_sample_by_id(DATASET, SAMPLE_ID)
        demo1_prediction_key = (DATASET, SAMPLE_ID, CAUSAL_PROMPT_VERSION, USE_RAG, RAG_MODE, RAG_TOP_K)
        demo1_prediction = generate(
            text=selected_sample["text"],
            sample_id=selected_sample["id"],
            client=client,
            retriever=retriever,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=CAUSAL_PROMPT_VERSION,
        )

    demo1_judgement = build_sample_judgement(
        prediction=demo1_prediction,
        gold=selected_sample,
        dataset=DATASET if MODE == "sample" else None,
    )

    print(f"Sample id: {selected_sample.get('id')}")
    print(f"Text: {selected_sample['text']}")
    print("Gold relations:")
    print(json.dumps(selected_sample.get("relations", []), indent=2, ensure_ascii=False))
    print("Pred triples:")
    print(json.dumps(demo1_prediction.get("triples", []), indent=2, ensure_ascii=False))
    print("Single-sample judgement:")
    print(json.dumps({
        "gold_has_causal": demo1_judgement["gold_has_causal"],
        "pred_has_causal": demo1_judgement["pred_has_causal"],
        "primary_metric": demo1_judgement["primary_metric"],
        "strict_token_f1_counts": demo1_judgement["strict_token_f1"]["counts"],
        "anchor_window_counts": demo1_judgement["anchor_window"]["counts"],
    }, indent=2, ensure_ascii=False))


In [ ]:
# Validate event-output parsing with fixed strings.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:

    test_cases = [
        '{"components": [{"role": "Action", "value": "killed", "attributes": []}, {"role": "Theme", "value": "troops", "attributes": []}]}',
        '```json\n{"components": []}\n```',
        'Here is the result:\n{"components": [{"role": "Theme", "value": "rain", "attributes": [{"role": "Attribute", "value": "heavy"}]}]}',
    ]
    for i, tc in enumerate(test_cases, 1):
        try:
            print(f"Case {i} passed: {parse_event_output(tc)}")
        except Exception as e:
            print(f"Case {i} failed: {e}")


In [ ]:
# Extract an event structure from one span.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:
    event = extract_event(
        span="the allocation of low cost ( RDP ) houses at Marikana West Extension 2",
        role="cause",
        client=client,
        prompt_version=EVENT_PROMPT_VERSION,
    )
    print(json.dumps(event, indent=2, ensure_ascii=False))


In [ ]:
# Assemble a complete knowledge-graph record.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:
    kg = build_kg_json(
        sample_id=None,
        text="5 Afghan troops killed after US army bombards warehouse in Kabul",
        triples=[
            {
                "cause": "US army bombards warehouse in Kabul",
                "effect": "5 Afghan troops killed",
                "relation": "caused",
            }
        ],
        client=client,
        prompt_version=EVENT_PROMPT_VERSION,
        triple_source="manual",
    )
    print(json.dumps(kg, indent=2, ensure_ascii=False))


In [ ]:
# Build a knowledge-graph record for the configured sample.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:

    current_kg = build_current_kg_json()
    print(json.dumps(current_kg, indent=2, ensure_ascii=False))
    print(f"causal_links: {len(current_kg['causal_links'])}")
    print(f"events: {len(current_kg['events'])}")


In [ ]:
# Convert the record to a NetworkX graph.
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:
    import networkx as nx
    from src.kg_builder import build_graph
    from src.kg_visualizer import visualize

    if "current_kg" not in globals():
        current_kg = build_current_kg_json()

    G = build_graph(current_kg)
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")

    print("\nNodes:")
    for node_id, data in G.nodes(data=True):
        role = data.get("role") or data.get("event_role")
        print(f"- {node_id}: type={data.get('type')}, role={role}, label={data.get('label')}")

    print("\nEdges:")
    for source, target, data in G.edges(data=True):
        print(f"- {source} -> {target}: {data.get('label')}")


In [ ]:
# Render the graph as interactive HTML.
import os
if RUN_MODE != "visualize":
    print("Skipping this single-sample visualization because RUN_MODE!='visualize'.")
else:
    from pathlib import Path
    from IPython.display import IFrame

    if "current_kg" not in globals():
        current_kg = build_current_kg_json()
    if "G" not in globals():
        G = build_graph(current_kg)

    source_label = "manual" if MODE == "manual" else f"{DATASET}_{SAMPLE_ID}_{TRIPLE_SOURCE}"
    html_path = visualize(G, str(PROJECT_ROOT / "outputs" / "kg_html" / f"{source_label}.html"))
    print(f"Visualization saved to: {html_path}")

    display_path = Path(html_path).resolve()
    try:
        iframe_src = Path(os.path.relpath(display_path, Path.cwd().resolve())).as_posix()
    except ValueError:
        iframe_src = display_path.as_uri()

    IFrame(iframe_src, width=900, height=600)


In [ ]:
# Rejudge cached extraction results without rerunning extraction.
if RUN_MODE != "rejudge" or not RUN_UNIT_REJUDGE:
    print("Cached unit rejudging is disabled; skipping.")
else:
    try:
        from tqdm.auto import tqdm as rejudge_tqdm
    except Exception:
        rejudge_tqdm = None

    rejudge_source = Path(REJUDGE_SOURCE_SPANS_PATH)
    if not rejudge_source.is_absolute():
        rejudge_source = PROJECT_ROOT / rejudge_source
    if not rejudge_source.exists():
        raise FileNotFoundError(f"Cached span file not found: {rejudge_source}")

    print(f"Cached extraction: {rejudge_source}")
    print(f"Judge: prompt={JUDGE_PROMPT_VERSION}; model={JUDGE_MODEL}; workers={JUDGE_MAX_WORKERS}")
    print("LM Studio / construction extraction: skipped")

    rejudge_config = KGRejudgeConfig(
        source_spans_path=rejudge_source,
        judge_prompt_version=JUDGE_PROMPT_VERSION,
        schema=EVAL_SCHEMA,
        max_depth=EVAL_MAX_DEPTH,
        output_dir=PROJECT_ROOT / REJUDGE_OUTPUT_DIR,
        judge_api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        judge_model=JUDGE_MODEL,
        judge_max_tokens=JUDGE_MAX_TOKENS,
        judge_timeout=JUDGE_TIMEOUT,
        judge_max_workers=JUDGE_MAX_WORKERS,
        judge_retry_times=JUDGE_RETRY_TIMES,
        judge_retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
        save_outputs=True,
    )
    rejudge_result = run_saved_kg_judging(rejudge_config, progress_factory=rejudge_tqdm)

    span_results = rejudge_result.span_results
    sample_results = rejudge_result.sample_results
    method_metrics = rejudge_result.method_metrics
    output_paths = rejudge_result.output_paths
    print("\n===== KG Rejudge Summary =====")
    for key, value in method_metrics.items():
        print(f"{key}: {value}")
    print("\nSaved files:")
    for name, path in output_paths.items():
        print(f"{name}: {path}")


In [ ]:
# Run independent global diagnostics on cached extraction results.
if RUN_MODE != "rejudge" or not RUN_GLOBAL_DIAGNOSTICS:
    print("Global diagnostics are disabled; skipping.")
else:
    try:
        from tqdm.auto import tqdm as global_tqdm
    except Exception:
        global_tqdm = None

    global_source = Path(REJUDGE_SOURCE_SPANS_PATH)
    if not global_source.is_absolute():
        global_source = PROJECT_ROOT / global_source
    cached_span_results = load_saved_span_results(global_source)
    global_records = prepare_global_diagnostic_records(
        cached_span_results,
        prompt_version=GLOBAL_DIAGNOSTIC_PROMPT_VERSION,
    )

    global_client = DeepSeekJudgeClient(
        api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        model=JUDGE_MODEL,
        max_tokens=JUDGE_MAX_TOKENS,
        timeout=JUDGE_TIMEOUT,
        retry_times=JUDGE_RETRY_TIMES,
        retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
    )
    global_progress = global_tqdm(total=len(global_records), desc="Global diagnostic spans") if global_tqdm else None
    try:
        global_records = run_parallel_global_diagnostics(
            global_records,
            global_client,
            max_workers=GLOBAL_DIAGNOSTIC_MAX_WORKERS,
            progress_callback=(lambda _record: global_progress.update(1)) if global_progress is not None else None,
        )
    finally:
        if global_progress is not None:
            global_progress.close()

    global_metrics = aggregate_global_diagnostics(global_records)
    global_dataset = str(cached_span_results[0].get("dataset", "unknown"))
    global_event_prompt = str(cached_span_results[0].get("prompt_version", "unknown"))
    global_sample_count = len({(item.get("dataset"), item.get("sample_id")) for item in cached_span_results})
    global_output_paths = save_global_diagnostic_outputs(
        global_records,
        global_metrics,
        output_dir=PROJECT_ROOT / GLOBAL_DIAGNOSTIC_OUTPUT_DIR,
        dataset=global_dataset,
        event_prompt_version=global_event_prompt,
        sample_count=global_sample_count,
        prompt_version=GLOBAL_DIAGNOSTIC_PROMPT_VERSION,
    )

    print("\n===== Independent Global Diagnostics =====")
    for key, value in global_metrics.items():
        display_value = "N/A" if value is None else value
        print(f"{key}: {display_value}")
    print("\nSaved global diagnostic files:")
    for name, path in global_output_paths.items():
        print(f"{name}: {path}")


## Evaluation protocol

Graph construction is evaluated on gold cause/effect spans so that causal-extraction errors are isolated. Deterministic checks cover JSON and schema validity, exact source substrings, forbidden causal roles, content-token coverage, and structural size and depth.

The semantic judge reports strict unit precision, correct unit yield, role correctness, attachment correctness, and judge coverage. A separate global diagnostic measures semantic-redundancy span rate and unsupported-hierarchy span rate. Metrics are aggregated from unit to span, sample, and method level.

Judge v3 is used for the main results. Scores from different rubric versions must be reported separately, together with both the construction-prompt and judge-prompt versions.


In [ ]:
# Evaluate graph construction on gold causal spans.
try:
    from tqdm.auto import tqdm
except Exception:
    class _NoProgress:
        def __init__(self, iterable=None, **_kwargs):
            self.iterable = [] if iterable is None else iterable

        def __iter__(self):
            return iter(self.iterable)

        def update(self, _value):
            return None

        def close(self):
            return None

    def tqdm(iterable=None, **kwargs):
        return _NoProgress(iterable, **kwargs)

if RUN_MODE != "evaluation":
    print("Skipping batch graph evaluation because RUN_MODE!='evaluation'.")
else:
    eval_output_dir = PROJECT_ROOT / "results" / "kg_evaluation"
    eval_schema = resolve_eval_schema(EVENT_PROMPT_VERSION, EVAL_SCHEMA)
    eval_max_depth = resolve_eval_max_depth(EVENT_PROMPT_VERSION, eval_schema, EVAL_MAX_DEPTH)
    eval_samples = load_dataset(EVAL_DATASET)
    eval_span_records = load_gold_span_records(
        eval_samples,
        dataset=EVAL_DATASET,
        sample_n=EVAL_SAMPLE_N,
        max_spans=EVAL_MAX_SPANS,
    )
    eval_gold_sample_count = len({record["sample_id"] for record in eval_span_records})

    print(f"Evaluation dataset: {EVAL_DATASET}")
    print(f"Prompt version: {EVENT_PROMPT_VERSION}")
    print(f"Schema: {eval_schema}; max_depth: {eval_max_depth}")
    print(f"Gold samples: {eval_gold_sample_count}")
    print(f"Gold spans: {len(eval_span_records)}")
    if RUN_JUDGE:
        print(f"Judge: prompt={JUDGE_PROMPT_VERSION}; model={JUDGE_MODEL} (thinking disabled); workers={JUDGE_MAX_WORKERS}; retries={JUDGE_RETRY_TIMES}")
    else:
        print("RUN_JUDGE=False: running extraction and deterministic validation only.")

    eval_config = KGEvalConfig(
        dataset=EVAL_DATASET,
        event_prompt_version=EVENT_PROMPT_VERSION,
        sample_n=EVAL_SAMPLE_N,
        max_spans=EVAL_MAX_SPANS,
        schema=EVAL_SCHEMA,
        max_depth=EVAL_MAX_DEPTH,
        extraction_retry_times=EVAL_EXTRACTION_RETRY_TIMES,
        checkpoint_every=EVAL_CHECKPOINT_EVERY,
        output_dir=eval_output_dir,
        run_judge=RUN_JUDGE,
        judge_prompt_version=JUDGE_PROMPT_VERSION,
        judge_api_key_path=PROJECT_ROOT / JUDGE_API_KEY_PATH,
        judge_model=JUDGE_MODEL,
        judge_max_tokens=JUDGE_MAX_TOKENS,
        judge_timeout=JUDGE_TIMEOUT,
        judge_max_workers=JUDGE_MAX_WORKERS,
        judge_retry_times=JUDGE_RETRY_TIMES,
        judge_retry_base_seconds=JUDGE_RETRY_BASE_SECONDS,
        save_outputs=True,
    )
    eval_result = run_kg_evaluation(
        config=eval_config,
        construction_client=client,
        samples=eval_samples,
        progress_factory=tqdm,
    )

    span_results = eval_result.span_results
    sample_results = eval_result.sample_results
    method_metrics = eval_result.method_metrics
    output_paths = eval_result.output_paths

    print("\n===== KG Evaluation Summary =====")
    for key, value in method_metrics.items():
        print(f"{key}: {value}")
    print("\nSaved files:")
    for name, path in output_paths.items():
        print(f"{name}: {path}")
